# **Hybrid Search RAG** using Langchain and Gemini

In [44]:
!pip install pypdf -q
!pip install google-genai -q
!pip install python-dotenv -q
!pip install numpy -q
!pip install scikit-learn -q

In [45]:
# Import necessary libraries
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

### Initialize Gemini LLM

In [ ]:
from google import genai
gemini_api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)
model_name = "gemini-2.5-flash"

### Initialize Embedding Model

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_embedding(text):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
    )
    return np.array(result.embeddings[0].values)

### Load PDF Document

In [48]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"D:\MyProjects\AI Project\Hybrid_Search_Rag_Langchain\codeprolk.pdf")

docs = loader.load()

### Split Documents into Chunks

In [ ]:
def chunk_text(text, chunk_size=250, overlap=30):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
    return chunks


all_chunks = []

for doc in docs:
    text = doc.page_content
    page = doc.metadata.get("page", None)

    doc_chunks = chunk_text(text)

    for chunk_text_content in doc_chunks:
        all_chunks.append({
            "content": chunk_text_content,
            "page": page
        })

In [50]:
len(all_chunks)

33

### Create Semantic Search Retriever

In [ ]:
import numpy as np
import time

print("Creating embeddings for chunks...")

chunk_embeddings = []

for i, chunk in enumerate(all_chunks):
    if i % 10 == 0:
        print(f"Processing chunk {i+1}/{len(all_chunks)}")

    text = chunk.get("content", "").strip()

    if not text:
        continue

    try:
        embedding = get_embedding(text)
        chunk_embeddings.append(embedding)

    except Exception as e:
        print(f"Error at chunk {i}: {e}")
        time.sleep(2)  # simple retry delay
        continue

chunk_embeddings = np.array(chunk_embeddings)

print(f"Created {len(chunk_embeddings)} embeddings")

Creating embeddings for chunks...
Processing chunk 1/33
Processing chunk 11/33
Processing chunk 21/33
Processing chunk 31/33
Created 33 embeddings


In [52]:
print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 33


### Create Keyword Search Retriever

In [ ]:
def keyword_search(query, chunks, k=2):
    query_lower = query.lower()
    query_words = set(query_lower.split())
    
    scores = []
    for chunk in chunks:
        chunk_lower = chunk['content'].lower()
        matches = len([w for w in query_words if w in chunk_lower])
        scores.append(matches)
    
    top_indices = np.argsort(scores)[-k:][::-1]
    return [chunks[i] for i in top_indices if scores[i] > 0]

In [54]:
print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

Chunk embeddings shape: (33, 3072)


### Create Hybrid Search Retriever

In [ ]:
def semantic_search(query, chunks, embeddings, k=2):
    query_embedding = get_embedding(query)
    similarities = cosine_similarity([query_embedding], embeddings)[0]
    
    top_indices = np.argsort(similarities)[-k:][::-1]
    return [chunks[i] for i in top_indices]

In [56]:
print("Retriever functions ready: keyword_search, semantic_search, hybrid_search")

Retriever functions ready: keyword_search, semantic_search, hybrid_search


### Define Prompt Template

In [ ]:
def hybrid_search(query, chunks, embeddings, k=2):
    semantic_results = semantic_search(query, chunks, embeddings, k)
    keyword_results = keyword_search(query, chunks, k)
    
    combined = {}
    for doc in semantic_results:
        combined[id(doc)] = {"doc": doc, "score": 1.0}
    for doc in keyword_results:
        doc_id = id(doc)
        if doc_id in combined:
            combined[doc_id]["score"] += 0.5
        else:
            combined[doc_id] = {"doc": doc, "score": 0.5}
    
    sorted_docs = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_docs[:k]]

### Create RAG Chain with Hybrid Search

In [ ]:
def rag_query(question, chunks, embeddings):

    relevant_chunks = hybrid_search(question, chunks, embeddings, k=2)
    context = "\n".join([chunk['content'] for chunk in relevant_chunks])
    
    prompt = f"""Answer this question using the provided context only.

Question: {question}

Context:
{context}
"""
    
    response = client.models.generate_content(
        model=model_name,
        contents=prompt
    )
    return response.text

### Invoke RAG Chain with Example Questions

In [65]:
response = rag_query("what are the popular videos in codeprolk", all_chunks, chunk_embeddings)
print(response)

The popular videos in CodePRO LK are:
1.  Python Basics: A series covering fundamental Python programming concepts.
2.  Machine Le


In [ ]:
for doc in keyword_search("what are the popular videos in codeprolk", all_chunks, k=2):
  print(doc['content'])
  print("---------------------")

=== Keyword Search Results ===

practical applications of theoretical concepts. 
• Tech Insights: Videos on the latest technology trends, best practices, and career advice. 
Popular Videos 
1. Python Basics: A series covering fundamental Python programming concepts. 
2. Machine Le
---------------------
ng how the videos have assisted them in their learning journeys. 
Impact 
The CodePRO LK YouTube channel has played a significant role in democratizing tech 
education in Sri Lanka. By providing free, high-quality educational content in Sinhala, it h
---------------------


In [62]:
print("=== Semantic Search Results ===")
for doc in semantic_search("what are the popular videos in codeprolk", all_chunks, chunk_embeddings, k=2):
  print(doc['content'])
  print("---------------------")

=== Semantic Search Results ===
d 
support each other. Additionally, the platform offers consultation services for personalized 
learning support. 
 
CodePRO LK YouTube Channel 
Overview 
The CodePRO LK YouTube Channel is a crucial extension of the platform, providing a wealth 
of 
---------------------

practical applications of theoretical concepts. 
• Tech Insights: Videos on the latest technology trends, best practices, and career advice. 
Popular Videos 
1. Python Basics: A series covering fundamental Python programming concepts. 
2. Machine Le
---------------------


In [63]:
print("=== Hybrid Search Results ===")
for doc in hybrid_search("what are the popular videos in codeprolk", all_chunks, chunk_embeddings, k=2):
  print(doc['content'])
  print("---------------------")

=== Hybrid Search Results ===

practical applications of theoretical concepts. 
• Tech Insights: Videos on the latest technology trends, best practices, and career advice. 
Popular Videos 
1. Python Basics: A series covering fundamental Python programming concepts. 
2. Machine Le
---------------------
d 
support each other. Additionally, the platform offers consultation services for personalized 
learning support. 
 
CodePRO LK YouTube Channel 
Overview 
The CodePRO LK YouTube Channel is a crucial extension of the platform, providing a wealth 
of 
---------------------
